<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase41_Google_Colab_Drive_Auto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 41 — E0–E7 synthetic challenge pack and cohort consequences

**Google Colab: Runtime → Run all. NO MANUAL FILE UPLOAD.** Google Drive will ask for authorization once. The notebook searches My Drive and accessible Shared drives for the **pinned Phase 39 source-ledger manifest + SQLite database** and the **completed Phase 40 JSON + evidence-gate CSV + logical mapping CSV**. Input files can be in different folders or have `(1)` or `-2` suffixes. Every selected file must pass SHA-256 and semantic checks. Outputs are generated on local Colab disk, then copied to `MyDrive/ICHI2027/Phase41/run_<UTC>/` with read-back hashes.

**What this phase does:** Creates E0–E7 *developer-authored synthetic* corruption scenarios against a clearly labeled **non-standard logical projection** of the existing 148 synthetic observations. It detects loss of anatomy/laterality, units, study/timepoint links, model version, a synthetic-only REVIEW_REQUIRED test state, duplicate records, and provenance links. It executes source and projected *synthetic proxy* cohort queries and records false inclusions/exclusions and root causes. It carries forward all 10 Arm A blockers from Phase 40.

**What it does NOT do:** It does not implement official OMOP or MI-CDM schemas, run official Arm A, compare frozen generic TFL v15, obtain any Athena approvals, use actual ADNI data, independently blind the test authors, or produce clinical MCI findings. **The original Phase 39 TRACE states are all PASS; the E5 REVIEW_REQUIRED source variant is fabricated for this synthetic challenge only and is never written to the immutable source database.** Success on the internal injector's known labels is engineering verification, not an unbiased sensitivity estimate.

Optional: set `PHASE41_DRIVE_FOLDER_HINT` to a relative folder (e.g. `Research/ICHI2027`) to narrow searching. For offline *testing only*, set `PHASE41_LOCAL_DRIVE_ROOT` to a local Drive mirror. Nothing private is ever requested or exported.

In [1]:
# CODE CELL 1/7 — Mount Google Drive and stage output locally
import os, re, csv, json, hashlib, sqlite3, shutil, tempfile, zipfile, time
import datetime as dt
from pathlib import Path
from collections import Counter, defaultdict

PROJECT = 'ICHI2027_neuroimaging_fhir_omop_fidelity'
PINNED_P39_MANIFEST = '381734533579c739336b3d99dd221cf2de5b37b174743b7cbf7bfdc50201f0c8'
PINNED_P40_GATES = '4c1febebd83ee969f9102499ddd09fba6542092c202fcfd7d98015797f1f454b'
PINNED_P40_CONTRACT = 'd80e7ede309fedca3e95341b299ae4756c9a6796ea2148c4abc0479f8ee235da'
# These are the exact CSV versions from the Phase 40 outputs supplied with this project.

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(2*1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

if os.environ.get('PHASE41_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE41_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    DRIVE_ROOTS=[MYDRIVE]
    print('Offline Drive-mirror test mode:',MYDRIVE)
else:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('Use Google Colab, or specify PHASE41_LOCAL_DRIVE_ROOT for an offline test.') from exc
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir(), 'Drive is mounted but MyDrive is inaccessible.'
    DRIVE_ROOTS=[MYDRIVE]
    if Path('/content/drive/Shareddrives').is_dir():
        DRIVE_ROOTS.append(Path('/content/drive/Shareddrives'))

hint=os.environ.get('PHASE41_DRIVE_FOLDER_HINT','').strip()
if hint:
    candidate=Path(hint).expanduser()
    if not candidate.is_absolute(): candidate=MYDRIVE/candidate
    if not candidate.is_dir(): raise FileNotFoundError('Folder hint not found: '+hint)
    DRIVE_ROOTS=[candidate.resolve()]

# All intermediate writes happen on local disk, not the Drive FUSE mount.
WORK=Path(tempfile.mkdtemp(prefix='ichi_phase41_'))
STAGE=WORK/'outputs'; STAGE.mkdir()
INPUT=WORK/'verified_inputs'; INPUT.mkdir()
BASE_OUT=Path(os.environ.get('PHASE41_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase41')))
RUN_TAG=dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
OUT=BASE_OUT/RUN_TAG
print('Drive search roots:',*[str(x) for x in DRIVE_ROOTS],sep='\n  ')
print('Planned Drive output folder:',OUT)

Mounted at /content/drive
Drive search roots:
  /content/drive/MyDrive
Planned Drive output folder: /content/drive/MyDrive/ICHI2027/Phase41/run_20260929T060837_721733Z


## 1 · Hash-locked automatic Drive discovery

Discovery searches filenames first and hashes only likely matches. A wrong duplicate in a closer folder cannot replace an approved input. Phase 40 JSON files with different run timestamps are compared on their **substantive results**; both uploaded `SHAREABLE` variants are acceptable only if their checked results agree. When the exact pinned files cannot be found, the notebook stops rather than requesting uploads or silently using a different experiment.

In [2]:
# CODE CELL 2/7 — Locate validated Phase 39 + Phase 40 evidence
REQUIRED={
 'p39_manifest':('ICHI2027_Phase39_SHAREABLE','.json'),
 'p39_db':('ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER','.sqlite'),
 'p40_summary':('ICHI2027_Phase40_SHAREABLE','.json'),
 'p40_gates':('ICHI2027_Phase40_ARM_A_EVIDENCE_GATES','.csv'),
 'p40_contract':('ICHI2027_Phase40_LOGICAL_MAPPING_CONTRACT','.csv'),
}

def matches(name,stem,ext,allow_dash=False):
    # Original, Colab downloads (1)/(2), and the user's SHAREABLE-2.json variant.
    extra=r'(?:\s*\(\d+\))?'
    if allow_dash: extra=r'(?:-\d+|\s*\(\d+\))?'
    return re.fullmatch(re.escape(stem)+extra+re.escape(ext),name,re.I) is not None

hits={k:[] for k in REQUIRED}
for root in DRIVE_ROOTS:
    for current, dirs, filenames in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for filename in filenames:
            for k,(stem,ext) in REQUIRED.items():
                if matches(filename,stem,ext,allow_dash=k=='p40_summary'):
                    hits[k].append(Path(current)/filename)
                    break
print('Candidate counts:',{k:len(v) for k,v in hits.items()})
missing=[k for k,paths in hits.items() if not paths]
if missing: raise FileNotFoundError('Required Phase 39/40 file(s) not found anywhere in searched Drive roots: '+', '.join(missing))

def choose_pinned(key,digest):
    matches_=[p for p in hits[key] if sha256(p)==digest]
    if not matches_:
        raise RuntimeError('No approved hash-matching '+key+' file. Expected '+digest+'; filenames found: '+str(len(hits[key])))
    return sorted(matches_,key=lambda p:str(p))[0]

P={}
P['p39_manifest']=choose_pinned('p39_manifest',PINNED_P39_MANIFEST)
M39=json.loads(P['p39_manifest'].read_text(encoding='utf-8'))
assert M39.get('phase')=='39_sql_source_ledger_cohort_replay_and_integrity'
P['p39_db']=choose_pinned('p39_db',M39['artifact_sha256']['ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER.sqlite'])
P['p40_gates']=choose_pinned('p40_gates',PINNED_P40_GATES)
P['p40_contract']=choose_pinned('p40_contract',PINNED_P40_CONTRACT)

# A timestamp is permitted to differ between reruns; scientific evidence fields may not.
EXPECTED40={
 'project':PROJECT,'phase':'40_source_evidence_gate_and_logical_mapping_contract',
 'scope':'PUBLIC_SYNTHETIC_SOURCE_SIDE_ONLY',
 'phase39_input_manifest_sha256':PINNED_P39_MANIFEST,
 'phase39_companion_sha256_verified':7,
 'source_db_integrity_check':'PASS','source_db_fk_check':'PASS',
 'synthetic_people':19,'synthetic_studies':38,'synthetic_observations':148,
 'synthetic_cohort_proxy_positive':15,'synthetic_cohort_proxy_negative':4,
 'clinical_mci_attested':False,'public_phase39_negative_control_report_checks':12,
 'synthetic_source_side_reconciliation':'PASS','logical_mapping_contract_rows':8,
 'private_evidence_receipt_status_counts':{'NOT_PROVIDED':7},
 'official_omop_etl_executed':False,'official_micdm_load_executed':False,
 'armA_status':'BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED','real_mri_rows_processed':0,
 'full_four_arm_evaluation':False,
}
valid40=[]
for path in hits['p40_summary']:
    try:
        obj=json.loads(path.read_text(encoding='utf-8'))
        if all(obj.get(k)==v for k,v in EXPECTED40.items()):
            valid40.append((path,obj))
    except (OSError,UnicodeError,ValueError): pass
if not valid40:
    raise RuntimeError('No Phase 40 SHAREABLE matched the supplied synthetic-only, 7-approval-missing state. '
                       'This notebook intentionally refuses to mix an unreviewed or changed Phase 40 run.')
# Same scientific contents, alternate run time: prefer latest time; break ties consistently.
P['p40_summary'],M40=sorted(valid40,key=lambda z:(z[1].get('run_utc',''),str(z[0])))[-1]

# Copy SQLite to a local disk before opening. Do not mutate the source in Drive.
DB_LOCAL=INPUT/'immutable_phase39_source.sqlite'
shutil.copyfile(P['p39_db'],DB_LOCAL)
assert sha256(DB_LOCAL)==M39['artifact_sha256']['ICHI2027_Phase39_SYNTHETIC_SOURCE_LEDGER.sqlite']
with P['p40_gates'].open(newline='',encoding='utf-8') as f: GATES=list(csv.DictReader(f))
with P['p40_contract'].open(newline='',encoding='utf-8') as f: CONTRACT=list(csv.DictReader(f))
assert len(GATES)==10 and all(r['official_validation_passed']=='False' for r in GATES)
assert len(CONTRACT)==8
assert all(r['phase40_evidence_receipt'] in ('NOT_PROVIDED','REQUIRES_FUTURE_EXECUTION') for r in GATES)
print('Verified exact SHA-256: Phase 39 manifest + SQLite and Phase 40 2 CSVs.')
print('Verified Phase 40 JSON scientific fields; alternate timestamp copies accepted:',len(valid40))

Candidate counts: {'p39_manifest': 1, 'p39_db': 1, 'p40_summary': 2, 'p40_gates': 2, 'p40_contract': 2}
Verified exact SHA-256: Phase 39 manifest + SQLite and Phase 40 2 CSVs.
Verified Phase 40 JSON scientific fields; alternate timestamp copies accepted: 2


## 2 · Build the synthetic-only logical projection and matched cohort query

The source SQLite file is opened **read-only**, checked for corruption and read into transient Python records. **This projection is not OMOP, not MI-CDM, and does not certify a target ETL.** It deliberately permits field omission/duplication in the *in-memory target copy* so detector tests can observe corruption. Cohort eligibility is the Phase 39 four-slot bilateral-hippocampal *synthetic proxy* (baseline and 12-month, left and right); MCI diagnoses are not attested. The qualified-evidence policy also excludes records with REVIEW_REQUIRED or missing TRACE state; E5 adds one separately labeled artificial REVIEW_REQUIRED source variant.

In [3]:
# CODE CELL 3/7 — Immutable source checks and projected-cohort SQL
from copy import deepcopy
conn=sqlite3.connect('file:'+str(DB_LOCAL)+'?mode=ro',uri=True)
assert conn.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
assert conn.execute('PRAGMA foreign_key_check').fetchall()==[]
conn.row_factory=sqlite3.Row
STUDIES={r['study_ref']:dict(r) for r in conn.execute('SELECT * FROM source_study')}
DEVICES={r['device_ref']:dict(r) for r in conn.execute('SELECT * FROM source_device')}
PROVS={r['obs_ref']:dict(r) for r in conn.execute('SELECT * FROM source_observation_provenance')}
SOURCE=[]
for raw in conn.execute('SELECT * FROM source_observation ORDER BY obs_ref'):
    o=dict(raw); dev=DEVICES[o['device_ref']]
    o.update(algorithm_name=dev['algorithm_name'],algorithm_version=dev['algorithm_version'])
    assert o['obs_ref'] in PROVS and PROVS[o['obs_ref']]['provenance_ref']==o['provenance_ref']
    SOURCE.append(o)
assert len(SOURCE)==148 and len(STUDIES)==38 and len(DEVICES)==19
assert Counter(x['trace_state'] for x in SOURCE)=={'PASS':148}
assert sum(1 for x in STUDIES.values() if x['uid_source_attested'])==0
conn.close()

# This SQLite layout is a TEST-HARNESS PROJECTION; official target schema columns are not represented.
DDL="""
CREATE TABLE projected_study (study_ref TEXT, person_ref TEXT, study_started TEXT);
CREATE TABLE projected_feature (
 obs_ref TEXT, person_ref TEXT, study_ref TEXT, biomarker_code TEXT,
 anatomy_source_code TEXT, laterality TEXT, timepoint TEXT,
 effective_utc TEXT, value_mm3 REAL, unit_code TEXT,
 algorithm_name TEXT, algorithm_version TEXT, trace_state TEXT,
 trace_version TEXT, device_ref TEXT, provenance_ref TEXT
);
"""
FIELDS=['obs_ref','person_ref','study_ref','biomarker_code','anatomy_source_code',
        'laterality','timepoint','effective_utc','value_mm3','unit_code',
        'algorithm_name','algorithm_version','trace_state','trace_version','device_ref','provenance_ref']
INSERT='INSERT INTO projected_feature ('+','.join(FIELDS)+') VALUES ('+','.join('?' for _ in FIELDS)+')'
SLOTS={('baseline','LEFT'),('baseline','RIGHT'),('12-month','LEFT'),('12-month','RIGHT')}

# The same strict SQL runs on reference source rows and projected target rows;
# synthetic source reference is deliberately never rewritten on Drive.
SQL_STRICT="""
WITH observed AS (
 SELECT f.*,s.person_ref AS linked_person,s.study_started AS linked_started,
        CASE WHEN f.unit_code='mm3' THEN f.value_mm3
             WHEN f.unit_code='cm3' THEN f.value_mm3*1000.0
             ELSE NULL END AS normalized_mm3
 FROM projected_feature f
 LEFT JOIN projected_study s ON f.study_ref=s.study_ref
 WHERE f.biomarker_code='hippocampus-volume'
), valid AS (
 SELECT *,CASE WHEN linked_person=person_ref AND linked_started=effective_utc
     AND normalized_mm3>0 AND provenance_ref IS NOT NULL
     AND ((laterality='LEFT' AND anatomy_source_code='SYN-HIP-LEFT')
       OR (laterality='RIGHT' AND anatomy_source_code='SYN-HIP-RIGHT'))
     AND timepoint IN ('baseline','12-month') THEN 1 ELSE 0 END AS valid_flag
 FROM observed
), slots AS (
 SELECT person_ref,timepoint,laterality,COUNT(*) AS n,
        SUM(valid_flag) AS nvalid,
        SUM(CASE WHEN trace_state IS NULL OR trace_state='REVIEW_REQUIRED' THEN 1 ELSE 0 END) AS unqualified
 FROM valid GROUP BY person_ref,timepoint,laterality
), eligible AS (
 SELECT person_ref FROM slots GROUP BY person_ref
 HAVING COUNT(*)=4 AND MIN(n)=1 AND MAX(n)=1
    AND SUM(nvalid)=4 AND SUM(unqualified)=0
)
SELECT person_ref FROM eligible ORDER BY person_ref
"""
# For the ordinary proxy, TRACE is not an eligibility criterion; it is a separate qualified-policy query.
SQL_PROXY=SQL_STRICT.replace(' AND SUM(unqualified)=0','')
# Intentionally lossy heuristic, NOT a real Arm B or recommended clinical query.
SQL_VALUE_ONLY="""
SELECT person_ref FROM projected_feature
WHERE biomarker_code='hippocampus-volume' AND value_mm3>0
 AND laterality IN ('LEFT','RIGHT') AND timepoint IN ('baseline','12-month')
GROUP BY person_ref HAVING COUNT(DISTINCT timepoint||'/'||laterality)=4
ORDER BY person_ref
"""
SQL_NAIVE_QUALIFIED="""
SELECT person_ref FROM projected_feature
WHERE biomarker_code='hippocampus-volume' AND value_mm3>0
 AND laterality IN ('LEFT','RIGHT') AND timepoint IN ('baseline','12-month')
GROUP BY person_ref
HAVING COUNT(DISTINCT timepoint||'/'||laterality)=4
 AND SUM(CASE WHEN COALESCE(trace_state,'PASS')='REVIEW_REQUIRED' THEN 1 ELSE 0 END)=0
ORDER BY person_ref
"""

def query_members(rows,sql):
    mem=sqlite3.connect(':memory:')
    try:
        mem.executescript(DDL)
        mem.executemany('INSERT INTO projected_study VALUES(?,?,?)',
                        [(r['study_ref'],r['person_ref'],r['study_started']) for r in STUDIES.values()])
        mem.executemany(INSERT,[tuple(r.get(k) for k in FIELDS) for r in rows])
        return {r[0] for r in mem.execute(sql)}
    finally: mem.close()

SOURCE_PROXY=query_members(SOURCE,SQL_PROXY)
SOURCE_QUALIFIED=query_members(SOURCE,SQL_STRICT)
assert len(SOURCE_PROXY)==15 and SOURCE_QUALIFIED==SOURCE_PROXY
print('Original immutable source cohort: 15 proxy-positive / 19 synthetic participants; all source TRACE states PASS.')

Original immutable source cohort: 15 proxy-positive / 19 synthetic participants; all source TRACE states PASS.


## 3 · Pre-registered developer-authored E0–E7 injections

The injection manifest and detector are separate functions. E2 includes a **valid mm³→cm³ normalization** that must not trigger a false alarm, as well as an independently injected missing-unit fault. E5 overlays one artificial REVIEW_REQUIRED state *only inside the synthetic test reference*. These cases are not externally authored/blinded; Phase 41 records their internally known truth without statistical claims about external performance.

In [4]:
# CODE CELL 4/7 — Inject controlled corruption into independent scenario copies

def pick(person,timepoint,region,side):
    found=[r for r in SOURCE if r['person_ref'].endswith('-'+person)
           and r['timepoint']==timepoint and r['biomarker_code']==region+'-volume' and r['laterality']==side]
    assert len(found)==1,(person,timepoint,region,side)
    return found[0]['obs_ref']

def item(rows,obs_ref):
    found=[x for x in rows if x['obs_ref']==obs_ref]
    assert len(found)==1,obs_ref
    return found[0]

# Ground truth is an injector-authored (obs_ref, event code) set, not an independent human review.
SCENARIOS={}
def new_scenario(name,description):
    SCENARIOS[name]={'description':description,'reference':deepcopy(SOURCE),
                     'projected':deepcopy(SOURCE),'truth':set(),'normalization_expected':set()}
    return SCENARIOS[name]

a=new_scenario('E0','Unchanged synthetic reference; no warnings expected')

a=new_scenario('E1','Entorhinal laterality altered while source anatomy remains left')
r=pick('p001','baseline','entorhinal','LEFT')
item(a['projected'],r)['laterality']='RIGHT'
a['truth'].add((r,'ANATOMY_LATERALITY_MISMATCH'))

a=new_scenario('E2','Valid cm3 normalization on one row; missing unit on another')
r=pick('p002','baseline','entorhinal','LEFT')
t=item(a['projected'],r);t['unit_code']='cm3';t['value_mm3']=t['value_mm3']/1000.0
a['normalization_expected'].add(r)
r=pick('p002','12-month','hippocampus','RIGHT')
item(a['projected'],r)['unit_code']=None
a['truth'].add((r,'UNIT_MISSING_OR_INCOMPATIBLE'))

a=new_scenario('E3','Hip baseline observation linked to its 12-month study')
r=pick('p003','baseline','hippocampus','LEFT')
t=item(a['projected'],r)
other=next(x for x in STUDIES.values() if x['person_ref']==t['person_ref'] and x['study_started']!=t['effective_utc'])
t['study_ref']=other['study_ref']
a['truth'].add((r,'STUDY_TIMEPOINT_LINK_MISMATCH'))

a=new_scenario('E4','Extraction-model version lost for one hippocampal observation')
r=pick('p004','baseline','hippocampus','LEFT')
item(a['projected'],r)['algorithm_version']=None
a['truth'].add((r,'MODEL_IDENTITY_LOSS'))

a=new_scenario('E5','Synthetic-only REVIEW_REQUIRED reference overlay stripped in logical projection')
r=pick('p005','baseline','hippocampus','LEFT')
item(a['reference'],r)['trace_state']='REVIEW_REQUIRED' # generated test variant, NOT original TRACE evidence
item(a['projected'],r)['trace_state']=None
a['truth'].add((r,'TRACE_STATE_LOSS'))

a=new_scenario('E6','Duplicated hippocampal feature and missing entorhinal study link')
r=pick('p006','baseline','hippocampus','LEFT')
a['projected'].append(deepcopy(item(a['projected'],r)))
a['truth'].add((r,'DUPLICATE_OBSERVATION'))
r=pick('p006','baseline','entorhinal','RIGHT')
item(a['projected'],r)['study_ref']=None
a['truth'].add((r,'STUDY_TIMEPOINT_LINK_MISMATCH'))

a=new_scenario('E7','Combined missing hippocampal unit and missing entorhinal provenance')
r=pick('p007','12-month','hippocampus','LEFT')
item(a['projected'],r)['unit_code']='arbitrary-unit'
a['truth'].add((r,'UNIT_MISSING_OR_INCOMPATIBLE'))
r=pick('p007','12-month','entorhinal','RIGHT')
item(a['projected'],r)['provenance_ref']=None
a['truth'].add((r,'PROVENANCE_LINK_LOSS'))
assert len(SCENARIOS)==8 and sum(map(lambda x:len(x['truth']),SCENARIOS.values()))==9
print('Prepared developer-authored scenarios E0–E7; 9 expected fault events plus 1 valid unit normalization.')

Prepared developer-authored scenarios E0–E7; 9 expected fault events plus 1 valid unit normalization.


## 4 · Run the loss detector against held-fixed source references

This is a **source-to-logical-projection comparison harness**. The detector does not see the injector's ground-truth labels. It uses immutable reference values, checks source/target row cardinality, compares normalized volume within 1e−6 mm³ absolute tolerance, validates anatomy/laterality and study links, and checks model/TRACE/provenance preservation. Phase 41 does not infer clinical diagnosis or recalculate TRACE from downstream data.

In [5]:
# CODE CELL 5/7 — Detector and strict internal engineering assertions

def detector(reference,projected):
    original={r['obs_ref']:r for r in reference}
    groups=defaultdict(list)
    for row in projected:groups[row['obs_ref']].append(row)
    alerts=set(); normalizations=set()
    for obs,source in original.items():
        dest=groups.get(obs,[])
        if len(dest)!=1:
            alerts.add((obs,'DUPLICATE_OBSERVATION' if len(dest)>1 else 'DROPPED_OBSERVATION'))
            # Compare the first duplicate to isolate cardinality without inventing additional faults.
            if not dest:continue
        t=dest[0]
        if t['person_ref']!=source['person_ref']:
            alerts.add((obs,'PERSON_LINK_MISMATCH'))
        if (t['anatomy_source_code']!=source['anatomy_source_code'] or
            t['laterality']!=source['laterality'] or
            t['biomarker_code']!=source['biomarker_code']):
            alerts.add((obs,'ANATOMY_LATERALITY_MISMATCH'))
        if (t['study_ref']!=source['study_ref'] or t['timepoint']!=source['timepoint']
            or t['effective_utc']!=source['effective_utc'] or
            not t['study_ref'] or t['study_ref'] not in STUDIES or
            STUDIES[t['study_ref']]['person_ref']!=t['person_ref'] or
            STUDIES[t['study_ref']]['study_started']!=t['effective_utc']):
            alerts.add((obs,'STUDY_TIMEPOINT_LINK_MISMATCH'))
        unit=t['unit_code'];val=t['value_mm3']
        if unit not in ('mm3','cm3') or not isinstance(val,(int,float)) or val<=0:
            alerts.add((obs,'UNIT_MISSING_OR_INCOMPATIBLE'))
        elif abs(float(val)*(1000 if unit=='cm3' else 1)-float(source['value_mm3']))>1e-6:
            alerts.add((obs,'VALUE_MISMATCH'))
        elif unit=='cm3' and source['unit_code']=='mm3':
            normalizations.add(obs)
        if t['algorithm_name']!=source['algorithm_name'] or t['algorithm_version']!=source['algorithm_version']:
            alerts.add((obs,'MODEL_IDENTITY_LOSS'))
        if t['trace_state']!=source['trace_state'] or t['trace_version']!=source['trace_version']:
            alerts.add((obs,'TRACE_STATE_LOSS'))
        if t['provenance_ref']!=source['provenance_ref'] or t['device_ref']!=source['device_ref']:
            alerts.add((obs,'PROVENANCE_LINK_LOSS'))
    for obs in set(groups)-set(original): alerts.add((obs,'UNEXPECTED_OBSERVATION'))
    return alerts,normalizations

CATALOG=[]; EVENTS=[]
for scenario,fixture in SCENARIOS.items():
    detected,norm=detector(fixture['reference'],fixture['projected'])
    expected=fixture['truth']
    # Internal tests catch both missed injected faults and false positives on clean/normalization rows.
    if detected!=expected or norm!=fixture['normalization_expected']:
        raise AssertionError(f'{scenario}: missed={sorted(expected-detected)}; false_alerts={sorted(detected-expected)}; '
                             f'expected_normalized={sorted(fixture["normalization_expected"])}; actual={sorted(norm)}')
    fixture['detected']=detected
    CATALOG.append({'scenario':scenario,'description':fixture['description'],
                    'synthetic_input_rows':len(fixture['projected']),
                    'ground_truth_fault_events':len(expected),'detected_fault_events':len(detected),
                    'internally_matched_events':len(expected&detected),
                    'false_alert_events':len(detected-expected),'valid_unit_normalizations':len(norm),
                    'engineer_authored_not_external':'True'})
    for obs,reason in sorted(detected):
        EVENTS.append({'scenario':scenario,'synthetic_obs_ref':obs,'reason_code':reason,
                       'is_injector_labeled':'True' if (obs,reason) in expected else 'False',
                       'ground_truth_origin':'DEVELOPER_AUTHORED_SYNTHETIC'})
assert len(EVENTS)==9 and not detector(SOURCE,SOURCE)[0]
print('Internal detector assertions PASSED: 8 controlled cases, 9/9 injected fault events, 0 unexpected alerts.')
print('One valid cm3 conversion normalized without an alert. NO independent validation claim.')

Internal detector assertions PASSED: 8 controlled cases, 9/9 injected fault events, 0 unexpected alerts.
One valid cm3 conversion normalized without an alert. NO independent validation claim.


## 5 · Quantify synthetic cohort consequences without mislabeling clinical disease

For each E0–E7 case, the notebook runs (1) the strict four-slot synthetic source reference, (2) the same strict query on the corrupted projection, (3) the qualified TRACE policy, and (4) a deliberately lossy value-only diagnostic. It outputs record/person denominators, false inclusions, false exclusions, and the synthetic person-level Jaccard overlap. E5 separately demonstrates why silently treating **missing** TRACE state as PASS creates a policy-level false inclusion. These are *controlled synthetic consequences*, **not** external evidence of clinical benefit.

In [6]:
# CODE CELL 6/7 — Paired synthetic membership tables and blocker carry-forward
COHORT=[]
def compare_members(ref,target):
    intersection=len(ref&target);union=len(ref|target)
    return {'source_n':len(ref),'target_n':len(target),
            'false_inclusions':len(target-ref),'false_exclusions':len(ref-target),
            'jaccard':round(intersection/union,6) if union else 1.0}
for scenario,fixture in SCENARIOS.items():
    src=fixture['reference'];tgt=fixture['projected']
    ref_proxy=query_members(src,SQL_PROXY)
    target_proxy=query_members(tgt,SQL_PROXY)
    ref_qualified=query_members(src,SQL_STRICT)
    target_qualified=query_members(tgt,SQL_STRICT)
    naive_proxy=query_members(tgt,SQL_VALUE_ONLY)
    naive_qualified=query_members(tgt,SQL_NAIVE_QUALIFIED)
    assert len(ref_proxy)==15
    if scenario=='E5':
        assert len(ref_qualified)==14 and len(target_qualified)==14 and len(naive_qualified)==15
    else:assert len(ref_qualified)==15
    for query_label,reference,result in [
       ('strict_proxy',ref_proxy,target_proxy),
       ('qualified_trace_policy',ref_qualified,target_qualified),
       ('value_only_diagnostic_not_armB',ref_proxy,naive_proxy),
       ('naive_trace_missing_as_pass_not_safe',ref_qualified,naive_qualified)]:
        row={'scenario':scenario,'query':query_label,'synthetic_source_people':19,
             'synthetic_source_observations':148,'projected_observation_rows':len(tgt),
             **compare_members(reference,result),'clinical_mci_attested':'False',
             'official_target_evaluation':'False'}
        COHORT.append(row)
    fixture['proxy_source']=ref_proxy;fixture['proxy_strict']=target_proxy
    fixture['qualified_source']=ref_qualified;fixture['qualified_strict']=target_qualified
assert len(COHORT)==32
assert len(SCENARIOS['E0']['proxy_strict'])==15
assert len(SCENARIOS['E1']['proxy_strict'])==15
assert len(SCENARIOS['E2']['proxy_strict'])==14
assert len(SCENARIOS['E3']['proxy_strict'])==14
assert len(SCENARIOS['E4']['proxy_strict'])==15
assert len(SCENARIOS['E6']['proxy_strict'])==14
assert len(SCENARIOS['E7']['proxy_strict'])==14
# No readiness changes: synthetic test completion cannot approve or execute official target work.
assert len(GATES)==10 and all(g['official_validation_passed']=='False' for g in GATES)
print('Source/reference vs projected cohort queries PASSED: 8 scenarios × 4 policy/diagnostic queries.')
for name in SCENARIOS:
    rows=[r for r in COHORT if r['scenario']==name and r['query']=='strict_proxy']
    print(name,': strict proxy target',rows[0]['target_n'],'of 15; false exclusions',rows[0]['false_exclusions'])
print('E5 generated-only REVIEW_REQUIRED: safe qualified 14; unsafe missing-as-PASS 15.')

Source/reference vs projected cohort queries PASSED: 8 scenarios × 4 policy/diagnostic queries.
E0 : strict proxy target 15 of 15; false exclusions 0
E1 : strict proxy target 15 of 15; false exclusions 0
E2 : strict proxy target 14 of 15; false exclusions 1
E3 : strict proxy target 14 of 15; false exclusions 1
E4 : strict proxy target 15 of 15; false exclusions 0
E5 : strict proxy target 15 of 15; false exclusions 0
E6 : strict proxy target 14 of 15; false exclusions 1
E7 : strict proxy target 14 of 15; false exclusions 1
E5 generated-only REVIEW_REQUIRED: safe qualified 14; unsafe missing-as-PASS 15.


## 6 · Save the public-safe Phase 41 outputs back to Google Drive

The notebook writes the catalog, detector events (synthetic IDs only), cohort-consequence table, unchanged Arm A blocker list, JSON summary and ZIP **locally first**, then copies each completed file to its timestamped Drive folder. It verifies every Drive read-back SHA-256. No Athena records, private review files, real MRI identifiers, or participant-level clinical data are read or published.

In [7]:
# CODE CELL 7/7 — Persist verified public outputs (local build → Drive copy → read-back SHA)

def write_csv(name,rows):
    assert rows
    p=STAGE/name
    with p.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader();w.writerows(rows)
    return p
CATALOG_CSV=write_csv('ICHI2027_Phase41_CHALLENGE_CATALOG.csv',CATALOG)
EVENTS_CSV=write_csv('ICHI2027_Phase41_DETECTION_EVENTS.csv',EVENTS)
COHORT_CSV=write_csv('ICHI2027_Phase41_COHORT_CONSEQUENCES.csv',COHORT)
GATES_CSV=write_csv('ICHI2027_Phase41_ARM_A_GATES_UNCHANGED.csv',GATES)
CONTRACT_CSV=write_csv('ICHI2027_Phase41_LOGICAL_CONTRACT_CARRIED_FORWARD.csv',CONTRACT)
README=STAGE/'ICHI2027_Phase41_README.txt'
README.write_text(
 'Phase 41: PUBLIC SYNTHETIC DEVELOPER-AUTHORED E0–E7 CHALLENGE PACK.\n'
 'Input: pinned Phase 39 source ledger + matching Phase 40 logical contract.\n'
 'All 148 original TRACE source states are PASS. E5 test-only REVIEW_REQUIRED was fabricated.\n'
 'Loss detector runs against a logical projection, NOT official OMOP or MI-CDM.\n'
 'Internal tests have known injected labels and are NOT independent, blinded validation.\n'
 'Proxy cohort of 15/19 is synthetic: NO SOURCE MCI DIAGNOSIS.\n'
 'No Athena data, official concept decisions, real DICOM UID, actual clinical Procedure, or real MRI.\n'
 'The original 10 Arm A gates remain unapproved; no frozen original TFL v15 arm C executed.\n'
 'Only synthetic public outputs from this folder belong in public GitHub.\n',encoding='utf-8')
output_csv=[CATALOG_CSV,EVENTS_CSV,COHORT_CSV,GATES_CSV,CONTRACT_CSV,README]
summary={
 'project':PROJECT,'phase':'41_synthetic_E0_E7_challenge_and_cohort_consequences',
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'scope':'PUBLIC_SYNTHETIC_DEVELOPER_AUTHORED_ONLY',
 'phase39_manifest_sha256':sha256(P['p39_manifest']),
 'phase39_immutable_sqlite_sha256':sha256(DB_LOCAL),
 'phase40_shareable_sha256':sha256(P['p40_summary']),
 'phase40_gates_csv_sha256':sha256(P['p40_gates']),
 'phase40_contract_csv_sha256':sha256(P['p40_contract']),
 'original_trace_source_states':{'PASS':148},
 'E5_review_required_source_variant':'ARTIFICIAL_SYNTHETIC_TEST_ONLY',
 'synthetic_people':19,'synthetic_studies':38,'synthetic_observations':148,
 'synthetic_source_proxy_positive':15,'clinical_mci_attested':False,
 'scenarios_executed':len(SCENARIOS),
 'developer_authored_injected_fault_events':sum(len(x['truth']) for x in SCENARIOS.values()),
 'internal_detected_labeled_fault_events':len(EVENTS),
 'unexpected_detector_events':0,
 'valid_unit_normalizations_without_alert':1,
 'cohort_query_rows':len(COHORT),
 'independently_blinded_external_validation':False,
 'official_omop_etl_executed':False,'official_micdm_load_executed':False,
 'frozen_original_generic_tfl_armC_executed':False,
 'real_mri_rows_processed':0,'full_A_B_C_D_evaluation':False,
 'armA_status':'BLOCKED_OFFICIAL_ARM_A_NOT_EXECUTED',
 'armA_unapproved_original_gates':len(GATES),
 'artifact_sha256':{p.name:sha256(p) for p in output_csv},
 'warning':'Logical projection and internally labeled stress tests are neither official ETL nor independent scientific validation.'
}
SUMMARY=STAGE/'ICHI2027_Phase41_SHAREABLE.json'
SUMMARY.write_text(json.dumps(summary,indent=2,sort_keys=True)+'\n',encoding='utf-8')
public=output_csv+[SUMMARY]
# The zipped artifact itself is not added to its contents; manifest records individual outputs.
ZIP=STAGE/'ICHI2027_Phase41_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in public:z.write(p,p.name)
assert len(zipfile.ZipFile(ZIP).namelist())==len(public)
OUT.mkdir(parents=True,exist_ok=False)
saved=[]
for source in public+[ZIP]:
    destination=OUT/source.name
    error=None
    for attempt in range(3):
        try:
            shutil.copyfile(source,destination)
            assert sha256(source)==sha256(destination),'Drive read-back SHA mismatch'
            saved.append(destination);error=None;break
        except (OSError,AssertionError) as exc:
            error=exc;print('Drive retry',attempt+1,'for',source.name,':',exc)
            time.sleep(1+attempt)
    if error is not None:
        raise RuntimeError('Outputs were generated locally but Drive save failed for '+source.name+
                           '; rerun only this final cell after fixing Drive. Local outputs: '+str(STAGE)) from error
assert len(saved)==len(public)+1
print('SUCCESS: 8 public synthetic Phase 41 files saved and hash-verified on Drive.')
print('Drive folder:',OUT)
print('Public ZIP:',OUT/ZIP.name)

SUCCESS: 8 public synthetic Phase 41 files saved and hash-verified on Drive.
Drive folder: /content/drive/MyDrive/ICHI2027/Phase41/run_20260929T060837_721733Z
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase41/run_20260929T060837_721733Z/ICHI2027_Phase41_PUBLIC_OUTPUTS.zip


## Next actual Arm A prerequisite — do not confuse Phase 41 with official target evidence

This phase deliberately proceeds with **public synthetic tests while private approvals remain missing**. To lift the Arm A block, obtain the authorized frozen Athena snapshot and valid manifest, independent decisions for all 19 bindings and the 36 original vocabulary gaps, an approved missing-demographics policy, exact approved OMOP/MI-CDM revision and DDL, and documented handling of source-absent clinical Procedures and real DICOM UIDs. Only then write and execute an **official-schema** Arm A loader and target FK/cardinality/SQL checks. Frozen unmodified generic TFL v15 and independently authored evaluation cases are separate later obligations; do not relabel these synthetic tests as completed A/B/C/D evaluation.